# AI Agent — Groq + ReAct + Tool Calling

A small ReAct-style AI agent built from scratch in Python:

- **LLM:** Groq (`llama-3.1-8b-instant`)
- **Tools:** Calculator (math via `eval`), Search (live web search via DuckDuckGo)
- **Pattern:** ReAct (Reason → Act → Observe, looped until a Final Answer)
- **Design:** `Tool` base class + `Agent` class, no global state

## Setup

1. `pip install -r requirements.txt`
2. Create a `.env` file in this directory containing:
   ```
   GROQ_API_KEY=your_key_here
   ```
3. Run all cells.

In [ ]:
import os
import json
import math

from dotenv import load_dotenv
from groq import Groq
from langchain_community.tools import DuckDuckGoSearchRun

load_dotenv()

## Tool Base Class

In [ ]:
class Tool:
    """Base class every tool inherits from. Guarantees a .name, .description, and .run() method,
    so the Agent can treat all tools the same way regardless of what they do internally."""

    def __init__(self, name, description):
        self.name = name
        self.description = description

    def run(self, tool_input):
        raise NotImplementedError("Each tool must implement its own run() method.")

## Tools: Calculator & Search

In [ ]:
class CalculatorTool(Tool):
    """Evaluates a math expression. Input MUST be a valid Python math expression made only of
    numbers, operators, and math functions (e.g. sqrt(81)) - never a sentence or description.
    Exposes every public name from the math module directly (sqrt, pow, log, pi, ...)."""

    def __init__(self):
        super().__init__(
            name="Calculator",
            description=(
                "Use for mathematical calculations. Input MUST be a valid Python math expression made only "
                "of numbers, operators (+ - * / ** //), and functions like sqrt(x), pow(x, y), log(x). "
                "Do the reasoning yourself first, then send ONLY the final numeric expression - e.g. send "
                "'4647000 * (5008/4647)', never a sentence describing the calculation."
            )
        )
        self.eval_namespace = {name: getattr(math, name) for name in dir(math) if not name.startswith("_")}
        self.eval_namespace["math"] = math

    def run(self, expression):
        try:
            return eval(expression, self.eval_namespace)
        except Exception:
            return "Calculation Error"


class SearchTool(Tool):
    """Live web search via DuckDuckGo."""

    def __init__(self):
        super().__init__(
            name="Search",
            description="Use for factual questions and information retrieval from the web."
        )
        self.search = DuckDuckGoSearchRun()

    def run(self, query):
        return self.search.run(query)

## The Agent

In [ ]:
class Agent:
    """A ReAct-style agent: loops Reason -> Act -> Observe until it produces a Final Answer
    or runs out of steps. Each instance owns its own client, tools, and state - no globals.

    Guardrails enforced in code (not just prompted), since a prompt alone isn't always reliably
    followed by a small/fast model:
      - each tool can be called at most `max_uses_per_tool` times per run
      - an exact repeat of a call (whether it ran or was blocked) is recognized and rejected
      - a Final Answer that's actually a raw, unevaluated math expression (e.g. "sqrt(81)") gets
        resolved into a real computed value before being accepted, reusing CalculatorTool's own
        eval_namespace as a verification step
    """

    def __init__(self, tools, model="llama-3.1-8b-instant", max_steps=10):
        groq_api_key = os.getenv("GROQ_API_KEY")
        self.client = Groq(api_key=groq_api_key)

        self.model = model
        self.max_steps = max_steps

        self.tools = {tool.name: tool for tool in tools}
        self.max_uses_per_tool = 2  # hard cap per tool per run, enforced in code

        self.state = {}
        self.system_prompt = self._build_system_prompt()

    def _build_system_prompt(self):

        tools_description = ""
        for i, tool in enumerate(self.tools.values(), start=1):
            tools_description += f"{i}. {tool.name}\n- {tool.description}\n\n"

        return f"""
You are an AI Agent that thinks step-by-step using the ReAct (Reason + Act) pattern.

Available Tools:

{tools_description}{len(self.tools) + 1}. Final Answer
- Use this ONLY when you already have enough information to fully answer the user's question.
- Your Final Answer input must be the actual resolved answer, in plain language - NEVER an unevaluated
  mathematical expression like 'sqrt(13109989)' or '4647000 * 1.08'. If a calculation is still needed,
  use Calculator first to get the real number, THEN give that number as your Final Answer.

You will be shown the conversation so far, including any earlier Thoughts, Actions, and Observations.
At every step, decide what to do next and return ONLY a JSON object (no markdown fences, no extra text)
in ONE of these two formats:

If you still need to use a tool:
{{
    "reason": "why you are taking this step",
    "tool": "the exact tool name",
    "input": "the input to give the tool"
}}

If you already know the final answer:
{{
    "reason": "why you now have enough information",
    "tool": "Final Answer",
    "input": "the final answer to give the user"
}}

Rules:
- Use only ONE tool per step.
- Always check the Observations already shown to you before acting - don't repeat a call you already made,
  and don't repeat a call you were already told you cannot make again.
- For Calculator specifically: input must be a valid Python math expression using ONLY numbers, operators,
  and math functions - never a sentence or description. Work out the numbers in your own reasoning first,
  then send just the final expression, e.g. '4647000 * (5008/4647)', not 'apply the growth rate to the
  population'.
- Real-world tool results are messy by nature: they mix old and current information, repeat the same fact
  in different words, and often report slightly different numbers from different sources - for example, a
  city's population vs. its district's population vs. its metro area's population, or a figure reported in
  different years or currencies. This is completely normal - do NOT keep using the same tool just to find
  one "perfect" or "exact" answer.
- If a name or fact appears more than once (even worded differently), treat it as confirmed. Each tool can
  only be used twice in this entire task - after that, you must switch tools or give a Final Answer.
- An approximate but timely answer is better than no answer. Give the Final Answer as soon as you reasonably
  can.
- You have a limited number of steps, and you will be told which step you're on. As you approach your last
  step, you MUST wrap up with a Final Answer using whatever information you've already gathered, even if it
  isn't perfect.
"""

    def _call_llm(self, messages):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0
        )
        return response.choices[0].message.content

    def run(self, query):

        self.state.clear()
        self.state["query"] = query
        self.state["steps"] = []
        self.state["result"] = None

        scratchpad = ""
        tool_usage_count = {}
        seen_calls = set()

        for step_number in range(self.max_steps):

            messages = [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": (
                    f"User Query: {query}\n\n{scratchpad}\n\n"
                    f"(You are on step {step_number + 1} of {self.max_steps} maximum steps.)\n\n"
                    "What is your next step?"
                )}
            ]

            llm_output = self._call_llm(messages)

            try:
                step_data = json.loads(llm_output)
            except Exception:
                print("Could not parse LLM output as JSON:", llm_output)
                break

            reason = step_data["reason"]
            tool_name = step_data["tool"]
            tool_input = step_data["input"]

            print(f"\n--- Step {step_number + 1} ---")
            print("Reason:", reason)
            print("Tool:", tool_name)
            print("Input:", tool_input)

            if tool_name == "Final Answer":
                final_text = str(tool_input)

                # safety net: if the model submitted a raw, unevaluated expression (e.g. "sqrt(13109989)")
                # instead of a real answer, resolve it for real using CalculatorTool's own eval_namespace.
                # eval() failing here is the NORMAL case (a real answer isn't valid Python), so we just
                # leave the text untouched whenever that happens.
                calculator = self.tools.get("Calculator")
                if calculator is not None:
                    try:
                        computed_value = eval(final_text, calculator.eval_namespace)
                        if isinstance(computed_value, (int, float)) and str(computed_value) != final_text.strip():
                            final_text = f"{tool_input} = {computed_value}"
                    except Exception:
                        pass

                self.state["result"] = final_text
                self.state["steps"].append({
                    "reason": reason, "tool": tool_name, "input": tool_input, "observation": "N/A"
                })
                print("\nFinal Answer:", final_text)
                return final_text

            call_signature = (tool_name, str(tool_input).strip().lower())

            if call_signature in seen_calls:
                observation = (
                    f"You already attempted this exact '{tool_name}' call with this input earlier in this "
                    f"task (whether it ran or was blocked) - repeating it again will not help. Try a "
                    f"meaningfully different input, switch to a different tool, or give your Final Answer now."
                )

            elif tool_usage_count.get(tool_name, 0) >= self.max_uses_per_tool:
                observation = (
                    f"You have already used '{tool_name}' {tool_usage_count[tool_name]} times in this task, "
                    f"which is the maximum allowed. Do NOT attempt to use '{tool_name}' again for the rest "
                    f"of this task. Switch to a different tool right now, or give your Final Answer "
                    f"immediately with your best estimate from what you've already gathered."
                )
                seen_calls.add(call_signature)

            else:
                tool = self.tools.get(tool_name)

                if tool is None:
                    observation = "Unknown tool requested."
                else:
                    observation = tool.run(tool_input)
                    tool_usage_count[tool_name] = tool_usage_count.get(tool_name, 0) + 1
                    seen_calls.add(call_signature)

            print("Observation:", observation)

            self.state["steps"].append({
                "reason": reason, "tool": tool_name, "input": tool_input, "observation": observation
            })

            scratchpad += f"\nThought: {reason}\nAction: {tool_name}\nAction Input: {tool_input}\nObservation: {observation}\n"

        print("\nAgent stopped: reached max_steps without a Final Answer.")
        return None

## Usage

In [ ]:
agent = Agent(tools=[CalculatorTool(), SearchTool()])

agent.run("What is 250 * 80?")

In [ ]:
agent.run("Who is the CEO of Google?")

In [ ]:
agent.run("Find Chennai's population and calculate the square root of it.")

In [ ]:
print(json.dumps(agent.state, indent=4))